In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import os
from copy import copy

import analysis_tools as tool
from config import *

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.patches import Rectangle
import matplotlib.patheffects as path_effects
import matplotlib.transforms as mtransforms

In [ ]:
grid_data = tool.load_grid_data()

# T2m temperature difference between CTL and dry run

In [ ]:
dts = pd.date_range(start="2021-07-13T00", end="2021-07-15T00", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"


ds_t2m_ctl = tool.read_merged_var_det("t2m", dts, f"{base_dir}/BLK_CTL/merged/t2m", accu=False)
ds_t2m_ens_ctl = tool.read_merged_var_ens("t2m", dts, f"{base_dir}/BLK_CTL/merged/t2m", accu=False)

ds_t2m_wlt = tool.read_merged_var_det("t2m", dts, f"{base_dir}/BLK_WLT/merged/t2m", accu=False)
ds_t2m_ens_wlt = tool.read_merged_var_ens("t2m", dts, f"{base_dir}/BLK_WLT/merged/t2m", accu=False)

ds_t2m_sat = tool.read_merged_var_det("t2m", dts, f"{base_dir}/BLK_SAT/merged/t2m", accu=False)
ds_t2m_ens_sat = tool.read_merged_var_ens("t2m", dts, f"{base_dir}/BLK_SAT/merged/t2m", accu=False)

In [ ]:
(ds_t2m_wlt - ds_t2m_ctl).sel(cell=grid_data["focus_cells_26_red"]).sel(step=ds_t2m_ctl["step"].dt.hour.isin(range(8,18))).mean()

In [ ]:
(ds_t2m_ens_wlt - ds_t2m_ens_ctl).sel(cell=grid_data["focus_cells_28"], step=ds_t2m_ens_ctl["step"].dt.hour.isin(range(8,18))).mean(dim=["step", "cell"]).std()

# Humidity

In [ ]:
dts = pd.date_range(start="2021-07-13T00", end="2021-07-15T21", freq="3h")
base_dir = "/automount/agh/s6tifohr/july21_eval/data"

ds_q_ctl = tool.read_merged_var_det("q", dts, f"{base_dir}/BLK_CTL/merged/q", accu=False)
ds_q_wlt = tool.read_merged_var_det("q", dts, f"{base_dir}/BLK_WLT/merged/q", accu=False)
ds_q_sat = tool.read_merged_var_det("q", dts, f"{base_dir}/BLK_SAT/merged/q", accu=False)

In [ ]:
hhl = xr.open_dataset("invar/det_hhl_dom1cut.grb", engine="cfgrib", backend_kwargs={"indexpath": None})["HHL"]
# Average adjacent half levels → full model level heights, assign generalVerticalLayer coord
hml = xr.DataArray(
    0.5 * (hhl.isel(generalVertical=slice(None, -1)).values + hhl.isel(generalVertical=slice(1, None)).values),
    dims=["generalVerticalLayer", "cell"],
    coords={"generalVerticalLayer": hhl["generalVertical"].values[:-1]},
)

hhl_ens = xr.open_dataset("invar/ens_hhl_dom2.grb", engine="cfgrib", backend_kwargs={"indexpath": None})["HHL"]
# Average adjacent half levels → full model level heights, assign generalVerticalLayer coord
hml_ens = xr.DataArray(
    0.5 * (hhl_ens.isel(generalVertical=slice(None, -1)).values + hhl_ens.isel(generalVertical=slice(1, None)).values),
    dims=["generalVerticalLayer", "cell"],
    coords={"generalVerticalLayer": hhl_ens["generalVertical"].values[:-1]},
)

In [ ]:
# Select only the levels present in the cut ds_q
hml_q = hml.sel(generalVerticalLayer=ds_q_ctl["generalVerticalLayer"])

ds_q_ctl_bl = ds_q_ctl.where(hml_q < 1500)#.sel(cell=grid_data["focus_cells_26_red"])
ds_q_wlt_bl = ds_q_wlt.where(hml_q < 1500)#.sel(cell=grid_data["focus_cells_26_red"])
ds_q_sat_bl = ds_q_sat.where(hml_q < 1500)#.sel(cell=grid_data["focus_cells_26_red"])

In [ ]:
(ds_q_wlt_1km.mean() - ds_q_ctl_1km.mean()) / ds_q_ctl_1km.mean()

In [ ]:
(ds_q_sat_1km.mean() - ds_q_ctl_1km.mean()) / ds_q_ctl_1km.mean()

In [ ]:
x0, x1, y0, y1 = PLOT_WINDOW
in_window = (
    (np.rad2deg(grid_data["grid_26_red"]["clon"].values) >= x0) &
    (np.rad2deg(grid_data["grid_26_red"]["clon"].values) <= x1) &
    (np.rad2deg(grid_data["grid_26_red"]["clat"].values) >= y0) &
    (np.rad2deg(grid_data["grid_26_red"]["clat"].values) <= y1)
)

data_ctl = ds_q_ctl.mean(dim=["step", "generalVerticalLayer"])
data_sat = ds_q_sat.mean(dim=["step", "generalVerticalLayer"])
data_wlt = ds_q_wlt.mean(dim=["step", "generalVerticalLayer"])

vmin = min(data_ctl[in_window].min(), data_sat[in_window].min(), data_wlt[in_window].min()).item()
vmax = max(data_ctl[in_window].max(), data_sat[in_window].max(), data_wlt[in_window].max()).item()

fig, axs = plt.subplots(1, 3, figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

for ax, data, title in zip(axs, [data_ctl, data_sat, data_wlt], ["CTL", "SAT", "WLT"]):
    im = ax.tricontourf(grid_data["tri_26_red"], data, vmin=vmin, vmax=vmax)
    ax.set(title=title)
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

fig.colorbar(im, ax=axs, orientation="horizontal", fraction=0.046, pad=0.04, label="q")
plt.show()

In [ ]:
dts = pd.date_range(start="2021-07-13T00", end="2021-07-15T21", freq="3h")
base_dir = "/automount/agh/s6tifohr/july21_eval/data"

ds_rh_ctl = tool.read_merged_var_det("rh", dts, f"{base_dir}/BLK_CTL/merged/rh", accu=False, format="netcdf")
ds_rh_ens_ctl = tool.read_merged_var_ens("rh", dts, f"{base_dir}/BLK_CTL/merged/rh", accu=False, format="netcdf")

ds_rh_wlt = tool.read_merged_var_det("rh", dts, f"{base_dir}/BLK_WLT/merged/rh", accu=False, format="netcdf")
ds_rh_ens_wlt = tool.read_merged_var_ens("rh", dts, f"{base_dir}/BLK_WLT/merged/rh", accu=False, format="netcdf")

ds_rh_sat = tool.read_merged_var_det("rh", dts, f"{base_dir}/BLK_SAT/merged/rh", accu=False, format="netcdf")
ds_rh_ens_sat = tool.read_merged_var_ens("rh", dts, f"{base_dir}/BLK_SAT/merged/rh", accu=False, format="netcdf")

In [ ]:
len(grid_data["focus_cells_28"])

In [ ]:
hml_ens.sel(generalVerticalLayer=ds_rh_ens_ctl["generalVerticalLayer"])

In [ ]:
hml_rh = hml.sel(generalVerticalLayer=ds_rh_ctl["generalVerticalLayer"], cell=grid_data["focus_cells_26_red"])
hml_rh_ens = hml_ens.sel(generalVerticalLayer=ds_rh_ens_ctl["generalVerticalLayer"], cell=grid_data["focus_cells_28"])

ds_rh_ctl_bl = ds_rh_ctl.where(hml_rh < 1500)
ds_rh_ens_ctl_bl = ds_rh_ens_ctl.where(hml_rh_ens < 1500)

ds_rh_wlt_bl = ds_rh_wlt.where(hml_rh < 1500)
ds_rh_ens_wlt_bl = ds_rh_ens_wlt.where(hml_rh_ens < 1500)

ds_rh_sat_bl = ds_rh_sat.where(hml_rh < 1500)
ds_rh_ens_sat_bl = ds_rh_ens_sat.where(hml_rh_ens < 1500)

In [ ]:
plt.plot(ds_rh_ctl_bl["step"], ds_rh_ctl_bl.mean(dim=["cell", "generalVerticalLayer"]), label="CTL", color=c_ctl)
plt.fill_between(ds_rh_ctl_bl["step"], 
                 ds_rh_ens_ctl_bl.mean(dim=["cell", "generalVerticalLayer"]).quantile(0.25, dim="mem"), 
                 ds_rh_ens_ctl_bl.mean(dim=["cell", "generalVerticalLayer"]).quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

plt.plot(ds_rh_wlt_bl["step"], ds_rh_wlt_bl.mean(dim=["cell", "generalVerticalLayer"]), label="WLT", color=c_dry)
plt.fill_between(ds_rh_wlt_bl["step"], 
                 ds_rh_ens_wlt_bl.mean(dim=["cell", "generalVerticalLayer"]).quantile(0.25, dim="mem"), 
                 ds_rh_ens_wlt_bl.mean(dim=["cell", "generalVerticalLayer"]).quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

plt.plot(ds_rh_sat_bl["step"], ds_rh_sat_bl.mean(dim=["cell", "generalVerticalLayer"]), label="SAT", color=c_wet)
plt.fill_between(ds_rh_sat_bl["step"], 
                 ds_rh_ens_sat_bl.mean(dim=["cell", "generalVerticalLayer"]).quantile(0.25, dim="mem"), 
                 ds_rh_ens_sat_bl.mean(dim=["cell", "generalVerticalLayer"]).quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

plt.legend()
plt.show()

In [ ]:
ds_rh_wlt_bl.mean() - ds_rh_ctl_bl.mean()

In [ ]:
(ds_rh_ens_wlt_bl.mean(dim=["step", "generalVerticalLayer", "cell"]) - ds_rh_ens_ctl_bl.mean(dim=["step", "generalVerticalLayer", "cell"])).mean()

In [ ]:
(ds_rh_ens_wlt_bl.mean(dim=["step", "generalVerticalLayer", "cell"]) - ds_rh_ens_ctl_bl.mean(dim=["step", "generalVerticalLayer", "cell"])).std()

# Boundary-layer state and the lifting condensation level

The parcel that sets the cloud base: its temperature, specific humidity and relative
humidity, and the resulting LCL and LFC. Mixed-layer parcel over the lowest 50 hPa,
thermodynamics by MetPy, focus-region land points, daytime of 13 July.

The chain to quote: specific humidity is nearly unchanged, the parcel is ~1.4 K warmer,
so its relative humidity falls and the LCL rises. The LFC follows from the LCL.

In [ ]:
import metpy.calc as mpcalc
from metpy.units import units

dts_conv = pd.date_range("2021-07-13T09", "2021-07-13T18", freq="3h")
base_dir = "/automount/agh/s6tifohr/july21_eval/data"

# model level heights and land mask for the focus region (both time invariant)
_hhl = xr.open_dataset("invar/det_hhl_dom1.grb", engine="cfgrib",
                       backend_kwargs={"indexpath": ""})["HHL"].values[:, grid_data["focus_cells_26"]]
z_full = (0.5 * (_hhl[:-1] + _hhl[1:]))[::-1]   # full levels, surface first
z_sfc = _hhl[-1]
is_land = xr.open_dataset("data/fr_land.grb", engine="cfgrib",
                          backend_kwargs={"indexpath": ""})["lsm"].values[grid_data["focus_cells_26"]] > 0.5


def parcel_diags(run_dir, dt):
    """Mixed-layer parcel (lowest 50 hPa): its T, q and RH, and its LCL and LFC in m above ground."""
    folder = dt if dt.hour % 3 == 0 else dt.ceil("3h")   # each cycle folder holds dt-2 ... dt
    ds = (xr.open_dataset(f"{base_dir}/{run_dir}/{folder:%Y%m%d%H}/fc_R03B07_rea_ml.{dt:%Y%m%d%H}",
                          engine="cfgrib",
                          backend_kwargs={"indexpath": "",
                                          "filter_by_keys": {"typeOfLevel": "generalVerticalLayer"}})
          .rename({"values": "cell"}).isel(cell=grid_data["focus_cells_26_red"])[["t", "q", "pres"]]
          .load().isel(generalVerticalLayer=slice(None, None, -1)))   # surface first

    p = ds["pres"].values * units.Pa
    T = ds["t"].values * units.K
    Td = mpcalc.dewpoint_from_specific_humidity(p, ds["q"].values * units("kg/kg"))

    n = p.shape[1]
    out = {k: np.full(n, np.nan) for k in ("T", "q", "rh", "lcl", "lfc")}
    for c in range(n):
        par_p, par_T, par_Td = mpcalc.mixed_parcel(p[:, c], T[:, c], Td[:, c], depth=50 * units.hPa)
        prof = mpcalc.parcel_profile(p[:, c], par_T, par_Td)
        lcl_p, _ = mpcalc.lcl(par_p, par_T, par_Td)
        lfc_p, _ = mpcalc.lfc(p[:, c], T[:, c], Td[:, c], prof)

        out["T"][c] = par_T.to("K").m
        out["q"][c] = mpcalc.specific_humidity_from_dewpoint(par_p, par_Td).m * 1e3    # g/kg
        out["rh"][c] = mpcalc.relative_humidity_from_dewpoint(par_T, par_Td).m * 1e2   # %
        # model pressure -> height, interpolated in log p, referenced to the ground
        lnp = -np.log(p[:, c].m)
        out["lcl"][c] = np.interp(-np.log(lcl_p.to("Pa").m), lnp, z_full[:, c]) - z_sfc[c]
        if np.isfinite(lfc_p.m):
            out["lfc"][c] = np.interp(-np.log(lfc_p.to("Pa").m), lnp, z_full[:, c]) - z_sfc[c]
    return out


res_conv = {r: {k: [] for k in ("T", "q", "rh", "lcl", "lfc")} for r in ("BLK_CTL", "BLK_WLT")}
for dt in dts_conv:
    for run in res_conv:
        d = parcel_diags(run, dt)
        for k in d:
            res_conv[run][k].append(d[k])

C = {k: np.array(res_conv["BLK_CTL"][k])[:, is_land] for k in res_conv["BLK_CTL"]}
W = {k: np.array(res_conv["BLK_WLT"][k])[:, is_land] for k in res_conv["BLK_WLT"]}
both = np.isfinite(C["lfc"]) & np.isfinite(W["lfc"])   # compare the LFC only where both runs have one

print(f"Focus-region land points, {dts_conv[0]:%d %b %H}-{dts_conv[-1]:%H} UTC, "
      f"50 hPa mixed-layer parcel, n = {C['T'].size} point-times\n")
for k, lab, unit in (("T", "temperature", "K"), ("q", "specific humidity", "g/kg"),
                     ("rh", "relative humidity", "%"), ("lcl", "LCL", "m")):
    c, w = np.nanmean(C[k]), np.nanmean(W[k])
    extra = (f"  (median {np.nanmedian(W[k] - C[k]):+.0f}, {100 * np.nanmean(W[k] > C[k]):.0f} % of points)"
             if k == "lcl" else "")
    print(f"  {lab:20s} CTL {c:7.2f}   WLT {w:7.2f}   change {w - c:+.2f} {unit}{extra}")
print(f"  {'LFC':20s} CTL {C['lfc'][both].mean():7.0f}   WLT {W['lfc'][both].mean():7.0f}   "
      f"change {(W['lfc'] - C['lfc'])[both].mean():+.0f} m  "
      f"(median {np.median((W['lfc'] - C['lfc'])[both]):+.0f}, "
      f"{100 * (W['lfc'] > C['lfc'])[both].mean():.0f} % of points; defined in both at {100 * both.mean():.0f} %)")

# Reviewer 2 Q: Precipitation bursts

In [ ]:
dts = pd.date_range(start="2021-07-13T00", end="2021-07-15T00", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

exp_name = "BLK_CTL"
da_ctl_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_ctl_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_SAT"
da_sat_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_sat_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

In [ ]:
# Compute time series of ensemble runs:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
ts_ctl_ens = ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).sum(dim="cell")
ts_sat_ens = ds_sat_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).sum(dim="cell")

In [ ]:
diff = (ts_sat_ens - ts_ctl_ens) / ts_ctl_ens * 100
diff = diff.sel(step=pd.date_range(start="2021-07-13T00", end="2021-07-15T00", freq="3h"))

besser Maxima nehmen?

In [ ]:
for mem in range(0,20):
    plt.plot(ts_ctl_ens["step"], (ts_sat_ens - ts_ctl_ens).sel(mem=mem))
    plt.axvline(np.datetime64("2021-07-14T00"))
    plt.axhline(0)
    plt.show()

In [ ]:
a = diff.sel(step=pd.date_range(start="2021-07-13T00", end="2021-07-14T00", freq="3h")).mean(dim="step")
a

In [ ]:
b = diff.sel(step=pd.date_range(start="2021-07-14T00", end="2021-07-15T00", freq="3h")).mean(dim="step")
b